# Time Series Forecasting with SAP HANA APL

**Automated Predictive Library (APL) — Step-by-Step Tutorial**

---

**APL (Automated Predictive Library)** is a component of SAP HANA that builds predictive models
automatically, directly inside the database. You do not need to be a data scientist to use it.
APL selects the best forecasting algorithm for your data, tunes its parameters, and returns
production-ready forecasts — all in a few lines of Python.

> **Note:** APL is the same engine that powers the time series forecasting
> capabilities in **SAP Analytics Cloud** (Smart Predict and Predictive Planning).

**What you will learn:**

- How to connect to SAP HANA from Python using a `ConnectionContext`
- The difference between a HANA DataFrame and a pandas DataFrame
- How to train a forecast model and generate predictions in a single call with `fit_predict()`
- How to control the forecast output columns with the `prediction_type` parameter
- How to explore results with an interactive HTML report (inline or as a shareable file)
- How to access model outputs programmatically — performance metrics, debrief reports, and more
- How to incorporate **future covariates** — additional variables known over the forecast horizon — to improve accuracy
- How to train one independent model per entity with **segmented modeling** ([Section 8](#section-8))
- How to persist a trained model to SAP HANA and reload it in a later session without retraining

## 1. Setup

We start by importing the required libraries and configuring the APL logger.

The APL logger controls how much internal detail APL prints during training. Setting it to
`logging.WARNING` keeps the output clean — only warnings
and errors will be shown. You can change it to `logging.INFO` for a full execution trace,
or to `logging.ERROR` to suppress warnings.

In [ ]:
import logging
import pandas as pd

import hana_ml
from hana_ml import dataframe as hd
from hana_ml.algorithms.apl.time_series import AutoTimeSeries
from hana_ml.algorithms.apl import apl_base
from hana_ml.model_storage import ModelStorage

# Keep APL log output clean — change to logging.INFO for verbose output
apl_base.config_logger(
    log_level=logging.WARNING,
    #log_path='C:/My_folder',
    #logfile_name='HANA_PYTHON_APL_TRACE',
)

print(f"hana_ml version: {hana_ml.__version__}")

## 2. Connect to SAP HANA

Open a `ConnectionContext` — the object that manages the database session. Two authentication
options are available:

- **Explicit credentials** — provide host, port, user, and password directly in the notebook.
- **User key** — reference a key stored in the SAP HANA secure user store (`hdbuserstore`).
  This is the recommended approach for shared or production notebooks because credentials are
  never written in plain text.

Everything that follows runs **inside SAP HANA**. The Python client only sends instructions and
receives results; the data and the model never leave the database unless you explicitly pull
them to the client.

Two types of DataFrames are used throughout this notebook:

- **HANA DataFrame** (`hana_ml.DataFrame`) — a lazy reference to a SQL query or table in SAP HANA.
  No data is transferred to the Python client; operations on it are translated into SQL and
  executed on the database.
- **pandas DataFrame** — an in-memory table on the Python client. You obtain one by calling
  `.collect()` on a HANA DataFrame, which executes the underlying query and pulls the result
  set to the client. Use `.collect()` only when you need to inspect or plot data locally.

In [ ]:
# Option 1 — explicit credentials
HDB_HOST =
HDB_PORT =
HDB_USER =
HDB_PASS =

conn = hd.ConnectionContext(
    HDB_HOST, HDB_PORT, HDB_USER, HDB_PASS, encrypt=True, sslValidateCertificate=False
)

# Option 2 — user key stored in hdbuserstore (recommended for shared/production notebooks)
# conn = hd.ConnectionContext(userkey='mykey', encrypt=True, sslValidateCertificate=False)

print("Connected to SAP HANA.")

## 3. The Dataset

The ozone dataset records monthly averages of hourly ozone (O3) readings in downtown Los Angeles from 1955 to 1972.
Each row has a date and a single numeric value.

This is a **univariate time series**: one measurement per time point, no additional explanatory
variables. APL can also handle multivariate series (with future covariates), but this dataset
lets us focus on the core forecasting workflow first.

The dataset is shipped with SAP HANA APL in the `APL_SAMPLES` schema, so no file upload is needed.
We simply point a HANA DataFrame at the existing table.

In [ ]:
DATE_COLUMN = "Date"
TARGET_COLUMN = "OzoneRateLA"
HORIZON = 12  # forecast 12 months ahead

# Create a HANA DataFrame pointing to the sample table — no upload needed
hana_df = conn.table("OZONE_RATE_LA", "APL_SAMPLES")

print(f"Rows: {hana_df.count()}")
print(f"Columns: {hana_df.columns}")
hana_df.tail(5).collect()

## 4. Train the Model and Generate Forecasts

We train the model in a single call using `fit_predict()`.
APL selects the best algorithm and returns fitted values plus a 12-step
forecast — all inside SAP HANA.

> **Note:** `fit_predict()` is convenient for exploration, but it does not retain a model
> binary that can be persisted. If you need to save the trained model and reuse it later
> (e.g. to generate forecasts in a separate session without retraining), use the
> `fit()` + `predict()` or `fit()` + `forecast()` workflows instead. See
> **Section 9 — Saving and Reloading Models** for details.

In [ ]:
model = AutoTimeSeries(
    time_column_name=DATE_COLUMN,
    target=TARGET_COLUMN,
    horizon=HORIZON,
)

forecast = model.fit_predict(data=hana_df, build_report=True)
print("Model training complete.")
print(f"Horizon-wide MAPE: {model.get_horizon_wide_metric('MAPE') * 100:.1f} %")

forecast.tail(HORIZON).collect()

Here is what each column contains:

| Column | Description |
|---|---|
| `Date` | Date of the data point. The last `horizon` rows are the future forecast steps; `ACTUAL` is `None` for those. |
| `ACTUAL` | Observed target value. `None` in forecast rows where no measurement exists yet. |
| `LOWER_INT_95PCT` | Lower bound of the 95% prediction interval. |
| `UPPER_INT_95PCT` | Upper bound of the 95% prediction interval. |
| `PREDICTED_1` | Forecast for this row's own date (horizon step 1). |
| `Trend_1` | Contribution of the **trend** component to `PREDICTED_1`. |
| `Cycles_1` | Contribution of the **seasonal cycle**. Negative values indicate periods that are seasonally below the trend line. |
| `ExtraPreds_1` | Contribution of **future covariates** (external predictors). `0.0` here — this is a univariate series with no covariates. |
| `Fluctuations_1` | Contribution of short-term **fluctuations** not captured by trend or cycles. |
| `Residues_1` | Difference between the actual and the fitted value for historical rows; `None` for future rows where the actual is unknown. |

The components add up to the point forecast:
`Trend_1 + Cycles_1 + ExtraPreds_1 + Fluctuations_1 = PREDICTED_1`.
The residual is excluded from this sum — it measures how far the historical fit is from the actual,
and is undefined for future rows. For a deeper understanding of the modeling technique and the role of each component, refer to the
[Time Series Modeling](https://help.sap.com/docs/apl/7223667230cb471ea916200712a9c682/774b073b60734d448dd010a8953beb20.html)
page in the APL documentation.

**Horizon-wide MAPE** is the Mean Absolute Percentage Error averaged across all horizon steps.
It measures how far off the model's predictions are, expressed as a percentage of the actual
values. It is the summary figure to use when communicating overall
model accuracy to stakeholders. Lower is better.

### Controlling the Output Format: `prediction_type`

`fit_predict()` accepts a `prediction_type` parameter that controls which columns are returned.

The example below uses `'Forecasts and Error Bars'` to obtain only the point forecast for the
nearest horizon step together with the 95% prediction interval bounds — without any component
decomposition.

In [ ]:
forecast_compact = model.fit_predict(
    data=hana_df,
    prediction_type="Forecasts and Error Bars",
)

print("Columns:")
print(forecast_compact.columns)
forecast_compact.tail(HORIZON).collect()

## 5. Explore the Interactive Report

Passing `build_report=True` to `fit_predict()` tells APL to compute all the debrief data
needed for the report during training — no extra database round-trip required afterward.

- `generate_notebook_iframe_report()` — renders the report **inline** in this notebook
- `generate_html_report(name)` — saves it as a **self-contained HTML file** (all charts and data included, no Python or SAP HANA required to view it) that you can share with colleagues or embed in a BI portal

The report has six tabs:

| Tab | What you see |
|---|---|
| **Overview** | Model metadata (build date, date/target variable, training window, horizon) and Horizon-Wide MAPE |
| **Forecast** | Line chart of actuals vs. in-sample fit and future forecast with 95 % confidence bands; table of forecast values; detected outliers |
| **Performance** | Metric tables (MAE, MAPE, SMAPE, RMSE, R², …) for the Validation and Estimation partitions |
| **Components** | Bar chart of the relative contribution of Trend, Cycle, and Residuals to total signal variance; breakdown chart of each component over time |
| **Cycles** | Seasonal profile: the impact (lift or drag) each period of the cycle (e.g. each calendar month) has on the forecast |
| **Local Explanations** | One waterfall chart per forecast horizon step showing the impact of each component on that specific prediction |

Several charts have a **Tips** button that displays a short explanation of the visualization.

In [ ]:
# Render the report inline in this notebook
model.generate_notebook_iframe_report()

# Save as a standalone HTML file you can share with stakeholders
model.generate_html_report("ozone_forecast_report")
print("Report saved to ozone_forecast_report.html")

## 6. Programmatic Access

The interactive report summarizes the model visually. This section shows how to access the
same information programmatically — useful when you need to extract specific numbers, log
results, or feed them into downstream systems.

Some information is available through direct functions on the model object; other details
are available through the debrief reports that APL can generate from the trained model.

### Model Components

In [ ]:
components = model.get_model_components()
print("Model components:")
for name, value in components.items():
    print(f"  {name}: {value}")

### Performance Metrics

`get_performance_metrics()` returns accuracy metrics evaluated on a **held-out validation
partition** — data that was not used for training. This gives an unbiased estimate of how
the model will perform on future data.

Metrics are reported **per forecast horizon**.

In [ ]:
metrics = model.get_performance_metrics()

df_metrics = pd.DataFrame(metrics, index=[f"Horizon {i + 1}" for i in range(HORIZON)])
df_metrics[["MAPE", "SMAPE", "RootMeanSquareError", "MASE"]].round(4)

`get_horizon_wide_metric()` collapses the per-horizon table into a single number by averaging
a given metric across all horizons.

In [ ]:
mean_mape = model.get_horizon_wide_metric("MAPE")
print(f"Horizon-wide MAPE: {mean_mape * 100:.1f} %")

### Debrief Reports

`get_debrief_report()` gives programmatic access to the individual statistical reports that APL
can generate from the trained model. Each report returns a HANA DataFrame that you can collect
and analyze.

The full list of available reports and their contents is documented in the
[APL Time Series Debrief Reports reference](https://help.sap.com/docs/apl/7223667230cb471ea916200712a9c682/8335ba9d1412442aa2031b6212d695de.html).

In [ ]:
overview = (
    model.get_debrief_report("TimeSeries_ModelOverview").deselect("Oid").collect()
)
print("Model Overview:")
overview

In [ ]:
perf = (
    model.get_debrief_report("TimeSeries_Performance")
    .deselect("Oid")
    .filter("\"Partition\" = 'Validation'")
    .collect()
)
print("Validation Performance:")
perf[["MAPE", "SMAPE", "RMSE", "MASE"]]

In [ ]:
outliers = model.get_debrief_report("TimeSeries_Outliers").deselect("Oid").collect()
if outliers.empty:
    print("No outliers detected.")
else:
    print(f"{len(outliers)} outlier(s) detected:")
    print(outliers.to_string(index=False))

In [ ]:
change_points = (
    model.get_debrief_report("TimeSeries_ChangePoints").deselect("Oid").collect()
)
if change_points.empty:
    print("No trend change points detected.")
else:
    print("Trend change points:")
    print(change_points.to_string(index=False))

In [ ]:
decomp = model.get_debrief_report("TimeSeries_Decomposition").deselect("Oid").collect()
print("Component Decomposition:")
decomp

## 7. Future Covariates

So far we have worked with a **univariate** series: one date column, one target, nothing else.
APL also supports **future covariates** — additional variables whose values are known for both
the entire training period and the entire forecast horizon at prediction time.

Typical examples are calendar effects (working-day count per month, public-holiday flags) or
planned business drivers (promotions, price changes). Because these values are known in advance,
APL can exploit them to improve forecast accuracy rather than treating them as random noise.

**How APL handles future covariates:**

- Any column in the training data that is neither the date column nor the target is automatically
  treated as a future covariate — no extra configuration required.
- At prediction time you must supply those same columns for the forecast period (the rows where
  the target is unknown). APL uses the covariate values to generate each forecast step.

**Dataset:** `APL_SAMPLES.CASHFLOWS_FULL` is a daily cash-flow series augmented with 23
calendar indicator columns. The table contains both the historical period (target known) and
the forecast period (target `NULL`), making it a self-contained illustration of the
future-covariates workflow.

In [ ]:
hana_cf = conn.table("CASHFLOWS_FULL", "APL_SAMPLES")

print(f"Total rows: {hana_cf.count()}")
print(f"Columns: {hana_cf.columns}")
hana_cf.head(5).collect()

The 21 rows where `Cash IS NULL` are the **forecast period**: the target is unknown but all
23 covariate values have already been computed from the calendar.

Because the full table already contains both periods, we can pass it directly to `fit_predict()`.
APL determines the training cut-off internally (the last row where the target is not null) and
uses the remaining rows as the forecast period, reading their covariate values to inform each prediction.

In [ ]:
n_train = hana_cf.filter('"Cash" IS NOT NULL').count()
n_forecast = hana_cf.filter('"Cash" IS NULL').count()
print(f"Training rows : {n_train}")
print(f"Forecast rows : {n_forecast}")
print()
print("First three forecast-period rows (covariate values are known, Cash is NULL):")
hana_cf.filter('"Cash" IS NULL').head(3).collect()

In [ ]:
model_cf = AutoTimeSeries(
    time_column_name="Date",
    target="Cash",
    horizon=21,  # matches the 21 forecast-period rows in CASHFLOWS_FULL
)

out_cf = model_cf.fit_predict(data=hana_cf, build_report=True)
print("Model training complete.")

model_cf.generate_notebook_iframe_report()
model_cf.generate_html_report("cashflow_forecast_report")
print("Report saved to cashflow_forecast_report.html")

Compared with the ozone report, the cashflows report has additional elements that reflect the contribution of the future covariates.

The component decomposition bar chart shows a fourth component: **Influencer: MondayMonthInd**. Here APL selected one of the 23 calendar columns as an influencer — a covariate whose values directly shift the forecast up or down. The bar shows its relative contribution to the total signal variance alongside Trend, Cycle, and Residuals. 

An **Influencers** tab appears when at least one future covariate is selected. Each influencer gets its own chart: the **x-axis** shows the covariate's possible values, the **y-axis** shows how much that value shifts the forecast up or down.

<a id="section-8"></a>

## 8. Segmented Modeling

So far the notebook has used single-series datasets: one time series, one model.
APL's `AutoTimeSeries` also supports **segmented modeling**: training one independent model
per segment in a single call.

**Why segmented modeling?**

- Different entities (stores, products, locations, …) have different patterns, seasonality,
  and trends. A single pooled model would average those differences away.
- Segmented modeling lets APL select the best algorithm and parameters **independently** for
  each entity, then return all forecasts in a single combined table.
- The same `fit_predict()` API is used — the only difference is the `segment_column_name`
  constructor argument.

**Dataset:** monthly recreation visits to US National Parks (1979–2019), published by the
National Park Service. The CSV is downloaded directly from GitHub and uploaded to a
SAP HANA table.

### Loading and Uploading the Dataset

The dataset comes from the National Park Service and records monthly recreation visits to
US National Parks from 1979 to 2019. Each row contains a park name, a date (first day of
the month), and the total visitor count for that month.

We download the CSV directly from GitHub using `pandas.read_csv()`, then upload it to a HANA table using `create_dataframe_from_pandas()`.

We keep five representative parks to keep training time short while still demonstrating the
segmented workflow.

In [ ]:
PARKS_TABLE = "NATIONAL_PARK_VISITS"
SEGMENT_COLUMN = "ParkName"

# Parse dates and rename columns to match expected names
df_parks = pd.read_csv(
    "https://raw.githubusercontent.com/antoinechabert/predictive/master/US%20National%20Park%20Recreation%20Visits.csv",
    sep=";",
)
df_parks["Date"] = pd.to_datetime(df_parks["Date"], format="%d/%m/%Y")

# Keep only a handful of parks to keep training fast
SELECTED_PARKS = [
    "Acadia NP",
    "Arches NP",
    "Grand Canyon NP",
    "Yellowstone NP",
    "Yosemite NP",
]
df_parks = df_parks[df_parks[SEGMENT_COLUMN].isin(SELECTED_PARKS)]

# Inject a broken segment (duplicate date) to demonstrate failure diagnosis below.
df_bad = pd.DataFrame(
    {
        SEGMENT_COLUMN: ["Broken Park"] * 4,
        "Date": [
            pd.Timestamp("2015-01-01"),
            pd.Timestamp("2015-02-01"),
            pd.Timestamp("2015-02-01"),  # duplicate — invalid for APL
            pd.Timestamp("2015-03-01"),
        ],
        "RecreationVisits": [100, 120, 130, 110],
    }
)
df_parks = pd.concat([df_parks, df_bad], ignore_index=True)

print(f"Rows loaded: {len(df_parks)}")
print(f"Parks: {sorted(df_parks[SEGMENT_COLUMN].unique())}")
df_parks.head(5)

In [ ]:
# Upload the pandas DataFrame to a HANA table
hana_parks = hd.create_dataframe_from_pandas(
    connection_context=conn,
    pandas_df=df_parks,
    table_name=PARKS_TABLE,
    force=True,  # overwrite if the table already exists
)
print(f"Data uploaded to HANA table: {PARKS_TABLE}")

### Segmented Modeling

Setting `segment_column_name` tells APL to train **one independent model per unique value
of that column** — here one model per park. All models are built in a single `fit_predict()`
call; APL handles the parallelism internally.

`max_tasks` controls how many models are trained (and forecast) in parallel. The default is
`1` (sequential). Setting it to `0` tells APL to use **all available HANA threads**, which
can significantly reduce wall-clock time when training many segments.

The forecast output includes the segment column so you can filter results by park.

> **Segmenting by multiple columns (e.g. Product family × Country)**
>
> `segment_column_name` accepts a single column. If your segments are defined by the
> combination of two or more dimensions, create a **composite key** column by concatenating
> them, then pass that column as the segment identifier:
>
> ```python
> df["Segment"] = df["ProductFamily"] + "_" + df["Country"]
> ```
>
> Each unique `ProductFamily_Country` pair becomes one independent model.

In [ ]:
HORIZON_PARKS = 12  # forecast 12 months ahead

model_parks = AutoTimeSeries(
    time_column_name="Date",
    target="RecreationVisits",
    horizon=HORIZON_PARKS,
    segment_column_name=SEGMENT_COLUMN,
    max_tasks=0,  # use all available HANA threads for parallel segment training
    force_positive_forecast=True,  # park visits can't be negative
)

forecast_parks = model_parks.fit_predict(
    data=hana_parks,
    prediction_type="First Forecast with Stable Components and Residues and Error Bars",
)

print("Training complete.")
print("Forecast preview for Grand Canyon NP:")
forecast_parks.filter(f"\"{SEGMENT_COLUMN}\" = 'Grand Canyon NP'").tail(
    HORIZON_PARKS
).collect()

### Generating the Model Report

In a segmented model each segment has its own independent model with its own forecast, seasonal
profile, and performance metrics. The report is therefore **per-segment**: you must specify
which segment to visualize via `build_report(segment_name=...)`.

One call to `build_report()` prepares the data for one segment. Call it again with a different
name to switch the report to another park.

In [ ]:
# build_report requires the caller to pick a segment; 1 report = 1 park
REPORT_SEGMENT = "Grand Canyon NP"

model_parks.build_report(segment_name=REPORT_SEGMENT)

# Display inline
model_parks.generate_notebook_iframe_report()

# Also save as a shareable HTML file
report_filename = (
    f"national_park_{REPORT_SEGMENT.replace(' ', '_').replace('/', '_')}_report"
)
model_parks.generate_html_report(report_filename)
print(f'Report for "{REPORT_SEGMENT}" saved to {report_filename}.html')

### Other Model Outputs

All programmatic outputs available for a single-series model are also available for a segmented
model. `get_performance_metrics()`, `get_model_components()`, and `get_debrief_report()` all
work the same way — the only difference is that the returned DataFrames include the segment
column, so you can filter down to a specific segment.

In [ ]:
model_parks.get_debrief_report("TimeSeries_Performance").filter(
    "\"Partition\" = 'Validation'"
).select("Oid", "MAPE", "RMSE", "MASE").collect().rename(columns={"Oid": "Segment"})

### Troubleshooting

Training and forecasting may succeed for some segments while failing for others. The overall
`fit_predict()` call does **not** raise an exception in that case — the successful segments
still return forecasts.

A common cause is an **invalid time series** for a specific segment. For example, a segment with duplicate timestamps cannot be modeled.

To illustrate this, the dataset above includes a synthetic **"Broken Park"** segment that has
a repeated date (`2015-02-01` appears twice), making it invalid for APL.

**Automatic warning**

When at least one segment fails, APL automatically emits a `WARNING` log message listing the
first 10 failed segments and the corresponding error. You can see this in the `fit_predict()`
output above. For a small number of failures this is usually sufficient to diagnose the problem.

**Checking task status per segment**

When there are more than 10 failures, the warning does not contain the full list. Use
`get_summary()` to retrieve all failed segments programmatically.
Filter on `AplTaskStatus` to get a quick overview of which segments succeeded and which failed:

In [ ]:
df = (
    model_parks.get_summary()
    .filter("\"KEY\" in ('AplTaskStatus')")
    .select("OID", "VALUE")
    .collect()
)
df.columns = [SEGMENT_COLUMN, "Task Status"]
df

**Inspecting the failure log for a specific segment**

For any segment that failed, `get_fit_operation_log()` gives the full APL log messages for
that segment.
Filter to `LEVEL = 0` (top-level messages) and the segment's `OID` to surface the root cause:

In [ ]:
df = (
    model_parks.get_fit_operation_log()
    .filter("LEVEL = 0 and OID = 'Broken Park'")
    .select("OID", "MESSAGE")
    .collect()
)
df.columns = [SEGMENT_COLUMN, "Log Text"]
df

## 9. Saving and Reloading Models

In production, you typically want to:

1. **Train** the model on a regular schedule (e.g. monthly)
2. **Save** it to a persistent HANA table
3. **Load** it later and generate forecasts without retraining

`ModelStorage` provides a central registry of all saved models. It stores the model binary,
its Python class, and its version history in a set of HANA tables.

Unlike `fit_predict()`, calling `fit()` retains the model binary inside SAP HANA, which is
required for persistence.

In [ ]:
model_fit = AutoTimeSeries(
    time_column_name=DATE_COLUMN,
    target=TARGET_COLUMN,
    horizon=HORIZON,
)

model_fit.fit(data=hana_df)
print("Model training complete.")
print(f"Horizon-wide MAPE: {model_fit.get_horizon_wide_metric('MAPE') * 100:.1f} %")

In [ ]:
MODEL_NAME = "Ozone Forecast Model"

model_storage = ModelStorage(
    connection_context=conn,
    schema="MODEL_STORAGE",
)

model_fit.name = MODEL_NAME

# Save — if_exists='replace' overwrites any existing model with the same name and version
model_storage.save_model(model=model_fit, if_exists="replace")
print(f'Model "{MODEL_NAME}" saved successfully.')

# Verify it appears in the registry
model_storage.list_models(name=MODEL_NAME)

In [ ]:
# Reload the model from the registry — simulates loading in a separate session
model_reloaded = model_storage.load_model(name=MODEL_NAME)

# Generate new forecasts with the reloaded model — no retraining needed
out_reloaded = model_reloaded.predict(data=hana_df)

print("Forecasts from the reloaded model:")
out_reloaded.tail(HORIZON).collect()

In [ ]:
# Clean up: remove the saved model from the registry
model_storage.delete_model(name=MODEL_NAME, version=1)
print(f'Model "{MODEL_NAME}" removed from storage.')

# Verify it is gone
remaining = model_storage.list_models(name=MODEL_NAME)
if remaining.empty:
    print("No models found — cleanup complete.")